# Phase 2: Full FT CaseHOLD + Random Label Baseline

**AutoDL H800 80GB** — 补充实验

| # | 实验 | 方法 | 数据集 | 模型 |
|---|------|------|--------|------|
| E9  | Full FT CaseHOLD Qwen  | Full FT | CaseHOLD | Qwen2.5-1.5B |
| E10 | Full FT CaseHOLD Llama | Full FT | CaseHOLD | Llama-3.2-1B |
| E11 | Random BillSum Qwen    | LoRA (shuffled labels) | BillSum  | Qwen2.5-1.5B |
| E12 | Random BillSum Llama   | LoRA (shuffled labels) | BillSum  | Llama-3.2-1B |
| E13 | Random CaseHOLD Qwen   | LoRA (shuffled labels) | CaseHOLD | Qwen2.5-1.5B |
| E14 | Random CaseHOLD Llama  | LoRA (shuffled labels) | CaseHOLD | Llama-3.2-1B |

## 0. 环境安装

In [ ]:
# AutoDL 网络加速
!source /etc/network_turbo

In [ ]:
# 安装依赖
!pip install torch transformers peft accelerate trl bitsandbytes datasets \
    huggingface_hub pyyaml rouge-score bert-score scikit-learn sentencepiece

In [ ]:
# HuggingFace 登录 (Llama 需要)
!huggingface-cli login

In [ ]:
import os
os.chdir('/root/MLP')
print('Working directory:', os.getcwd())

In [ ]:
# 创建符号链接 — 将持久化数据盘映射到项目目录
import os, subprocess

links = {
    '/root/autodl-tmp/outputs': '/root/MLP/outputs',
    '/root/autodl-tmp/logs':    '/root/MLP/logs',
    '/root/autodl-tmp/data':    '/root/MLP/data',
}

for target, link in links.items():
    os.makedirs(target, exist_ok=True)
    if os.path.islink(link):
        print(f'  ✓ symlink exists: {link} -> {os.readlink(link)}')
    elif os.path.exists(link):
        print(f'  ⚠ {link} is a real dir, skipping (move contents manually if needed)')
    else:
        os.symlink(target, link)
        print(f'  ✓ created: {link} -> {target}')

print('\nDone.')

## 1. 数据验证

In [ ]:
import json
from pathlib import Path

def count_jsonl(path):
    """Count valid JSONL records and detect corrupt lines."""
    p = Path(path)
    if not p.exists():
        return None, None
    good, bad = 0, 0
    with open(p, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                json.loads(line)
                good += 1
            except json.JSONDecodeError:
                bad += 1
    return good, bad

# BillSum
billsum_files = [
    'data/billsum/train_sft.jsonl',
    'data/billsum/val_sft.jsonl',
    'data/billsum/test_us_sft.jsonl',
    'data/billsum/test_ca_sft.jsonl',
]

# CaseHOLD
casehold_files = [
    'data/casehold/train_mc.jsonl',
    'data/casehold/validation_mc.jsonl',
    'data/casehold/test_mc.jsonl',
]

print('=== Data Validation ===')
all_ok = True
for f in billsum_files + casehold_files:
    good, bad = count_jsonl(f)
    if good is None:
        print(f'  ✗ MISSING: {f}')
        all_ok = False
    elif bad > 0:
        print(f'  ⚠ {f}: {good:,} ok, {bad} corrupt')
        all_ok = False
    else:
        print(f'  ✓ {f}: {good:,} records')

if all_ok:
    print('\n✅ All data files present and valid.')
else:
    print('\n❌ Issues found — fix before training.')
    print('  如果文件不存在，先跑预处理:')
    print('  python src/data/billsum/run_preprocessing.py')
    print('  python src/data/casehold/run_casehold_preprocessing.py')

In [ ]:
# 查看数据样例
import json

def peek(path, n=1):
    print(f'\n--- {path} ---')
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            rec = json.loads(line)
            print(f'  input:  {rec["input"][:200]}...')
            print(f'  output: {rec["output"][:200]}')

peek('data/billsum/train_sft.jsonl')
peek('data/casehold/train_mc.jsonl')

---
## 2. Full Fine-Tuning — CaseHOLD

使用 `train.py`（无 `lora` 节时自动全量微调）。  
Config 已更新：`bf16=true`, `adamw_torch`, `batch_size=4`, `grad_accum=4`。

### E9: Full FT — CaseHOLD × Qwen2.5-1.5B

In [ ]:
!python src/train/train.py --config configs/full_casehold_qwen.yaml 2>&1 | tee logs/full_casehold_qwen.log

In [ ]:
# E9 推理
!python src/evaluate/inference.py \
    --config configs/full_casehold_qwen.yaml \
    --split test \
    --batch_size 16

In [ ]:
# E9 评估
!python src/evaluate/eval_casehold.py \
    --predictions outputs/full_casehold_qwen/predictions_test.jsonl \
    --output results/casehold/full_qwen_test.json

### E10: Full FT — CaseHOLD × Llama-3.2-1B

In [ ]:
!python src/train/train.py --config configs/full_casehold_llama.yaml 2>&1 | tee logs/full_casehold_llama.log

In [ ]:
# E10 推理
!python src/evaluate/inference.py \
    --config configs/full_casehold_llama.yaml \
    --split test \
    --batch_size 16

In [ ]:
# E10 评估
!python src/evaluate/eval_casehold.py \
    --predictions outputs/full_casehold_llama/predictions_test.jsonl \
    --output results/casehold/full_llama_test.json

---
## 3. Random Label Baseline (LoRA)

**原理：** 将训练集的 output 打乱（input 不变），使 input→output 映射完全随机。  
如果模型确实学到了任务知识，random label 训练后在真实测试集上的表现应大幅低于正常训练。

- BillSum: 打乱 summary（output 字段）
- CaseHOLD: 打乱答案字母（output 字段）

### 3.0 生成 Random Label 数据

In [ ]:
import json
import random
from pathlib import Path

random.seed(42)

def shuffle_labels(input_path, output_path):
    """Load JSONL, shuffle the 'output' field across records, write new JSONL."""
    records = []
    with open(input_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    # Extract and shuffle outputs
    outputs = [r['output'] for r in records]
    random.shuffle(outputs)

    # Reassign shuffled outputs
    for rec, new_out in zip(records, outputs):
        rec['output'] = new_out

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')

    print(f'  ✓ {output_path}: {len(records):,} records (labels shuffled)')

print('=== Generating Random Label Datasets ===')

# BillSum
shuffle_labels('data/billsum/train_sft.jsonl',  'data/billsum/train_sft_random.jsonl')
shuffle_labels('data/billsum/val_sft.jsonl',    'data/billsum/val_sft_random.jsonl')

# CaseHOLD
shuffle_labels('data/casehold/train_mc.jsonl',        'data/casehold/train_mc_random.jsonl')
shuffle_labels('data/casehold/validation_mc.jsonl',   'data/casehold/validation_mc_random.jsonl')

print('\nDone. Test files are NOT shuffled (evaluate on real data).')

In [ ]:
# 验证：random label 数据 vs 原始数据的 output 不匹配率
import json

def mismatch_rate(orig_path, random_path):
    with open(orig_path) as f1, open(random_path) as f2:
        orig = [json.loads(l)['output'] for l in f1 if l.strip()]
        rand = [json.loads(l)['output'] for l in f2 if l.strip()]
    mismatches = sum(a != b for a, b in zip(orig, rand))
    return mismatches / len(orig)

print('Mismatch rates (should be ~1.0 for BillSum, ~0.8 for CaseHOLD 5-class):')
print(f'  BillSum train:     {mismatch_rate("data/billsum/train_sft.jsonl", "data/billsum/train_sft_random.jsonl"):.4f}')
print(f'  BillSum val:       {mismatch_rate("data/billsum/val_sft.jsonl", "data/billsum/val_sft_random.jsonl"):.4f}')
print(f'  CaseHOLD train:    {mismatch_rate("data/casehold/train_mc.jsonl", "data/casehold/train_mc_random.jsonl"):.4f}')
print(f'  CaseHOLD val:      {mismatch_rate("data/casehold/validation_mc.jsonl", "data/casehold/validation_mc_random.jsonl"):.4f}')

### E11: Random Label LoRA — BillSum × Qwen2.5-1.5B

In [ ]:
!python src/train/train.py --config configs/random_billsum_qwen.yaml 2>&1 | tee logs/random_billsum_qwen.log

In [ ]:
# E11 推理 test_us
!python src/evaluate/inference.py \
    --config configs/random_billsum_qwen.yaml \
    --split test_us \
    --batch_size 8

In [ ]:
# E11 推理 test_ca
!python src/evaluate/inference.py \
    --config configs/random_billsum_qwen.yaml \
    --split test_ca \
    --batch_size 8

In [ ]:
# E11 评估
!python src/evaluate/eval_billsum.py \
    --predictions outputs/random_billsum_qwen/predictions_test_us.jsonl \
    --output results/billsum/random_qwen_test_us.json

!python src/evaluate/eval_billsum.py \
    --predictions outputs/random_billsum_qwen/predictions_test_ca.jsonl \
    --output results/billsum/random_qwen_test_ca.json

### E12: Random Label LoRA — BillSum × Llama-3.2-1B

In [ ]:
!python src/train/train.py --config configs/random_billsum_llama.yaml 2>&1 | tee logs/random_billsum_llama.log

In [ ]:
# E12 推理 test_us
!python src/evaluate/inference.py \
    --config configs/random_billsum_llama.yaml \
    --split test_us \
    --batch_size 8

In [ ]:
# E12 推理 test_ca
!python src/evaluate/inference.py \
    --config configs/random_billsum_llama.yaml \
    --split test_ca \
    --batch_size 8

In [ ]:
# E12 评估
!python src/evaluate/eval_billsum.py \
    --predictions outputs/random_billsum_llama/predictions_test_us.jsonl \
    --output results/billsum/random_llama_test_us.json

!python src/evaluate/eval_billsum.py \
    --predictions outputs/random_billsum_llama/predictions_test_ca.jsonl \
    --output results/billsum/random_llama_test_ca.json

### E13: Random Label LoRA — CaseHOLD × Qwen2.5-1.5B

In [ ]:
!python src/train/train.py --config configs/random_casehold_qwen.yaml 2>&1 | tee logs/random_casehold_qwen.log

In [ ]:
# E13 推理
!python src/evaluate/inference.py \
    --config configs/random_casehold_qwen.yaml \
    --split test \
    --batch_size 16

In [ ]:
# E13 评估
!python src/evaluate/eval_casehold.py \
    --predictions outputs/random_casehold_qwen/predictions_test.jsonl \
    --output results/casehold/random_qwen_test.json

### E14: Random Label LoRA — CaseHOLD × Llama-3.2-1B

In [ ]:
!python src/train/train.py --config configs/random_casehold_llama.yaml 2>&1 | tee logs/random_casehold_llama.log

In [ ]:
# E14 推理
!python src/evaluate/inference.py \
    --config configs/random_casehold_llama.yaml \
    --split test \
    --batch_size 16

In [ ]:
# E14 评估
!python src/evaluate/eval_casehold.py \
    --predictions outputs/random_casehold_llama/predictions_test.jsonl \
    --output results/casehold/random_llama_test.json

---
## 4. Results Summary

In [ ]:
import json
from pathlib import Path

def load_result(path):
    p = Path(path)
    if not p.exists():
        return None
    with open(p) as f:
        return json.load(f)

# ── CaseHOLD: Full FT vs LoRA vs QLoRA ──
print('=' * 60)
print('CaseHOLD — Accuracy (n=5,221)')
print('=' * 60)
print(f'{"Method":<20} {"Model":<20} {"Accuracy":>10}')
print('-' * 50)

casehold_results = [
    ('LoRA',           'Qwen2.5-1.5B', 'results/casehold/lora_qwen_test.json'),
    ('LoRA',           'Llama-3.2-1B', 'results/casehold/lora_llama_test.json'),
    ('Full FT',        'Qwen2.5-1.5B', 'results/casehold/full_qwen_test.json'),
    ('Full FT',        'Llama-3.2-1B', 'results/casehold/full_llama_test.json'),
    ('QLoRA 4-bit',    'Qwen2.5-1.5B', 'results/casehold/qlora_qwen_test.json'),
    ('QLoRA 4-bit',    'Llama-3.2-1B', 'results/casehold/qlora_llama_test.json'),
    ('Random (LoRA)',  'Qwen2.5-1.5B', 'results/casehold/random_qwen_test.json'),
    ('Random (LoRA)',  'Llama-3.2-1B', 'results/casehold/random_llama_test.json'),
]

for method, model, path in casehold_results:
    r = load_result(path)
    acc = f'{r["accuracy"]:.4f}' if r else '—'
    print(f'{method:<20} {model:<20} {acc:>10}')

# ── BillSum: LoRA vs Full FT vs Random ──
print()
print('=' * 70)
print('BillSum — ROUGE-2 / BERTScore-F1')
print('=' * 70)
print(f'{"Method":<20} {"Model":<18} {"US R-2":>8} {"CA R-2":>8} {"US BERT":>8} {"CA BERT":>8}')
print('-' * 70)

billsum_results = [
    ('LoRA',          'Qwen2.5-1.5B', 'results/billsum/lora_qwen_test_us.json', 'results/billsum/lora_qwen_test_ca.json'),
    ('LoRA',          'Llama-3.2-1B', 'results/billsum/lora_llama_test_us.json', 'results/billsum/lora_llama_test_ca.json'),
    ('Full FT',       'Qwen2.5-1.5B', 'results/billsum/full_qwen_test_us.json', 'results/billsum/full_qwen_test_ca.json'),
    ('Full FT',       'Llama-3.2-1B', 'results/billsum/full_llama_test_us.json', 'results/billsum/full_llama_test_ca.json'),
    ('Random (LoRA)', 'Qwen2.5-1.5B', 'results/billsum/random_qwen_test_us.json', 'results/billsum/random_qwen_test_ca.json'),
    ('Random (LoRA)', 'Llama-3.2-1B', 'results/billsum/random_llama_test_us.json', 'results/billsum/random_llama_test_ca.json'),
]

for method, model, us_path, ca_path in billsum_results:
    us = load_result(us_path)
    ca = load_result(ca_path)
    us_r2   = f'{us["rouge2"]["mean"]:.4f}' if us else '—'
    ca_r2   = f'{ca["rouge2"]["mean"]:.4f}' if ca else '—'
    us_bert = f'{us["bertscore_f1"]["mean"]:.4f}' if us and 'bertscore_f1' in us else '—'
    ca_bert = f'{ca["bertscore_f1"]["mean"]:.4f}' if ca and 'bertscore_f1' in ca else '—'
    print(f'{method:<20} {model:<18} {us_r2:>8} {ca_r2:>8} {us_bert:>8} {ca_bert:>8}')

print('\nDone. Commit results: git add results/ && git commit -m "results: phase 2" && git push')

## 5. 提交结果

In [ ]:
!git add results/
!git commit -m "results: add Full FT CaseHOLD + random label baseline results"
!git push origin main